In [1]:
import os
import pickle
import numpy as np
import faiss
import torch
from transformers import AutoProcessor, CLIPModel
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from IPython.display import display, HTML
from PIL import Image
import base64
from io import BytesIO

In [2]:
INDEX_DIRECTORY = "faiss_index"
MODEL_NAME = "openai/clip-vit-base-patch32"

In [3]:
%%time
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = CLIPModel.from_pretrained(MODEL_NAME)
model.to(device)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


CPU times: user 716 ms, sys: 431 ms, total: 1.15 s
Wall time: 6.39 s


CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPSdpaAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e

In [4]:
%%time
# load vectors
index_path = os.path.join(INDEX_DIRECTORY, 'video_index.faiss')
index = faiss.read_index(index_path)
print(f"Loaded FAISS index with {index.ntotal} vectors")

# load metadata
# Load metadata
metadata_path = os.path.join(INDEX_DIRECTORY, 'video_metadata.pkl')
with open(metadata_path, 'rb') as f:
    metadata = pickle.load(f)
print(f"Loaded metadata for {len(metadata)} videos")

Loaded FAISS index with 1472 vectors
Loaded metadata for 1472 videos
CPU times: user 13.3 ms, sys: 6.03 ms, total: 19.3 ms
Wall time: 112 ms


In [5]:
def encode_text(text_query):
    """Encode text query using CLIP model."""
    inputs = processor(text=[text_query], return_tensors="pt", padding=True, truncation=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        text_features = model.get_text_features(**inputs)
    
    # Convert to numpy and normalize
    text_embedding = text_features.cpu().numpy().astype(np.float32)
    faiss.normalize_L2(text_embedding)
    return text_embedding

In [6]:
def search_videos(text_query, top_k=10):
    """Search for videos based on text query."""
    print(f"Searching for: '{text_query}'")
    
    # Encode query
    query_embedding = encode_text(text_query)
    
    # Search
    scores, indices = index.search(query_embedding, top_k)
    
    # Format results
    results = []
    for i, (score, idx) in enumerate(zip(scores[0], indices[0])):
        if idx != -1:
            results.append({
                'rank': i + 1,
                'score': float(score),
                'video_id': metadata[idx]['video_id'],
                'video_path': metadata[idx]['video_path'],
                'thumbnail_path': metadata[idx]['thumbnail_path']
            })
    
    return results

In [7]:
def display_thumbnails_grid(results, columns=3):
    total = len(results)
    rows = (total + columns - 1) // columns  # Ceiling division to get number of rows

    fig, axes = plt.subplots(rows, columns, figsize=(columns * 5, rows * 5))
    axes = axes.flatten()  # Flatten in case of single row

    for idx, result in enumerate(results):
        thumbnail_path = result['thumbnail_path']
        # img = Image.open(thumbnail_path).resize(thumbnail_size)
        img = Image.open(thumbnail_path)

        axes[idx].imshow(img)
        axes[idx].axis('off')
        axes[idx].set_title(f"Rank {result['rank']} | ID {result['video_id']}")

    # Hide any empty subplots if total is not a multiple of columns
    for idx in range(total, len(axes)):
        axes[idx].axis('off')

    plt.tight_layout()
    plt.show()

In [8]:
def image_to_base64(image_path):
    with Image.open(image_path) as img:
        buffered = BytesIO()
        img.save(buffered, format="JPEG")
        img_str = base64.b64encode(buffered.getvalue()).decode()
        return img_str

def display_clickable_thumbnails(results, columns=3):
    html = '<table style="border-collapse: collapse;">'
    for i in range(0, len(results), columns):
        html += '<tr>'
        for result in results[i:i+columns]:
            video_id = result['video_id']
            thumbnail_path = result['thumbnail_path']
            video_path = result['video_path']

            # Construct localhost URL path
            filename = os.path.basename(video_path)
            video_url = f"http://localhost:8000/{filename}"

            # Convert image to base64
            img_base64 = image_to_base64(thumbnail_path)

            # Add cell with image and hyperlink
            cell_html = f'''
                <td style="padding:10px; text-align:center;">
                    <a href="{video_url}" target="_blank">
                        <img src="data:image/jpeg;base64,{img_base64}" style="display:block; margin:auto; border:2px solid #000;" />
                    </a>
                    <div style="margin-top:5px;">Rank {result['rank']} | ID {video_id}</div>
                    <div><a href="{video_url}" target="_blank">Link</a></div>
                </td>
            '''
            html += cell_html
        html += '</tr>'
    html += '</table>'

    display(HTML(html))

In [22]:
%%time
results = search_videos("rahul gandhi", 10)
# results

# fitness
# cats
# cooking
# dancing
# standup comedy
# adventure
# talking head
# home decor

Searching for: 'rahul gandhi'
CPU times: user 95.2 ms, sys: 192 μs, total: 95.4 ms
Wall time: 28.9 ms


In [24]:
# display_thumbnails_grid(results, columns=3)
# display_clickable_thumbnails(results, columns=3)